# Comparing stocks

One page across every stock run with `python run.py`, built from the summaries each report saved in `reports/summaries/`. It has the same two parts as the reports:

- **Part 1, direction:** up or down tomorrow, judged against "always up."
- **Part 2, move size:** big or small move tomorrow, judged against the "Recent swing" rule (with "same as today" for context).

**The lucky-winners problem.** Test enough stocks and models, and some will "win" by chance alone. At the usual 5% level, about 1 comparison in 20 passes even when nothing real is going on. So each part shows two bars:

- **5%:** the usual threshold for a single test.
- **Corrected:** the *Holm correction*. Sort the p-values from smallest to largest: the smallest must clear 5% divided by the number of comparisons, the next 5% divided by one fewer, and so on, stopping at the first that misses. A result has to clear this to count as evidence once many things were tested. The two parts answer different questions, so each is corrected on its own.

Every accuracy is compared with its baseline using a **paired test**: a baseline predicts the same days as the model, so only the days where they disagree count. The 95% ranges in the charts come from the same days.

In [ ]:
# Settings (run.py replaces these)
TICKERS = ["SPY"]

In [ ]:
import sys
from pathlib import Path

# Find the project folder, whether this runs from the project, notebooks/ or reports/
ROOT = Path.cwd()
if ROOT.name in ("notebooks", "reports"):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from IPython.display import Markdown, display

pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams["figure.figsize"] = (11, 4)
# Same colors in every report: blue, orange, aqua, yellow
plt.rcParams["axes.prop_cycle"] = cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100"])

import numpy as np
from src.report import comparison_table, load_summaries, lstm_comparison

summaries = load_summaries(TICKERS)
dir_table, dir_info = comparison_table(summaries, task="direction")
vol_table, vol_info = comparison_table(summaries, task="volatility")
print(f"Stocks: {', '.join(TICKERS)}")

MODEL_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]


def plot_edges(table, edge_column, baseline, title):
    """Each model's accuracy minus the baseline's, with the 95% range of that difference."""
    plot = table.reset_index()
    models = list(dict.fromkeys(plot["model"]))
    stocks = list(dict.fromkeys(plot["stock"]))
    colors = dict(zip(models, MODEL_COLORS))
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(stocks) * len(models) + 1.5))
    y, ticks, labels = 0, [], []
    for stock in stocks:
        for model in models:
            row = plot[(plot["stock"] == stock) & (plot["model"] == model)].iloc[0]
            ax.errorbar(row[edge_column] * 100, y, xerr=row[f"{edge_column} ±"] * 100, fmt="o", color=colors[model],
                        markersize=8, capsize=4, linewidth=2, label=model if stock == stocks[0] else None)
            ticks.append(y)
            labels.append(f"{stock}  {model}")
            y += 1
        y += 0.5
    ax.axvline(0, color="grey", linewidth=1)
    ax.set_yticks(ticks, labels)
    ax.invert_yaxis()
    ax.set_xlabel(f'accuracy minus "{baseline}" (percentage points)')
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.3)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=len(models), frameon=False)
    plt.tight_layout()
    plt.show()


def luck_summary(info, accuracy_test, luck_test):
    """How many passes there are, next to how many luck alone would produce."""
    n, n_luck = info["comparisons"], info["luck comparisons"]
    display(Markdown(f"""
{len(TICKERS)} stocks; {n} stock-and-model pairs.

| Test | Comparisons | Passes expected by luck at 5% | Strictest corrected bar (Holm) | Passes at 5% | Passes after correction |
|---|---|---|---|---|---|
| {accuracy_test} | {n} | about {info["expected chance passes"]:.1f} | p-value at most {info["corrected threshold"]:.4f} | {info["accuracy passes (5%)"]} of {n} | {info["accuracy passes (corrected)"]} of {n} |
| {luck_test} | {n_luck} | about {info["expected chance luck passes"]:.1f} | luck share at most {info["luck corrected threshold"]:.4f} | {info["luck passes (5%)"]} of {n_luck} | {info["luck passes (corrected)"]} of {n_luck} |
"""))

# Part 1: direction

## 1. Test periods

Each stock is tested from its own first test year, so periods can differ for stocks with shorter histories.

In [ ]:
pd.DataFrame([
    {"stock": s["ticker"], "test start": s["test start"], "test end": s["test end"],
     "test days": s["test days"], "always up accuracy": s["always up accuracy"],
     "buy and hold annual return": s["buy and hold"]["annual return"]}
    for s in summaries
]).set_index("stock")

## 2. Accuracy against "always up"

**Edge** is the model's accuracy minus the "always up" accuracy for the same stock and days. Zero means no better than always predicting up. The bars show the 95% range of that difference, from the paired counts: if a bar crosses zero, the edge could be luck.

In [ ]:
plot_edges(dir_table, "edge", "always up", 'Direction: model edge over "always up", with 95% range')

## 3. Full table

- **edge ±:** half the width of the edge's 95% range.
- **p-value vs always up:** the paired test: the chance of being right on this many of the days the model said "down", if it were no better than "always up."
- **share of random >=:** from the luck check: how often random cash days did at least as well in the backtest, on annual return. Empty when a model never left the market.
- The four True/False columns: does the result pass at 5%, and does it pass the corrected bar?

In [ ]:
dir_table

## 4. How many wins would luck produce?

In [ ]:
luck_summary(dir_info, 'Accuracy beats "always up"', "Backtest beats luck (annual return)")
display(Markdown("A handful of passes at 5%, close to what luck predicts, is **not** evidence of skill. "
                 "Only results that also pass the corrected bar, and that hold up on future data "
                 "(phase 6, the live forward test), deserve attention."))

# Part 2: move size

## 5. Baselines and buy-and-hold for each stock

- **share of big days:** how often tomorrow's move beat the stock's typical day over the past year.
- **majority:** accuracy of always predicting whichever is more common.
- **same as today:** accuracy of predicting tomorrow will be big if today's move was bigger than usual.
- **recent swing:** accuracy of predicting a big day when today's high-to-low swing was wider than usual for the past year. This is the bar for Part 2.
- **buy and hold:** the Sharpe ratio and worst drop that the half-on-big strategy tries to improve on, next to the Recent swing rule's own weekly half-on-big strategy.

In [ ]:
pd.DataFrame([
    {"stock": s["ticker"], "test start": s["volatility"]["test start"], "test days": s["volatility"]["test days"],
     "share of big days": s["volatility"]["share of big days"],
     "majority": s["volatility"]["majority accuracy"],
     "same as today": s["volatility"]["same as today accuracy"],
     "recent swing": s["volatility"]["recent swing accuracy"],
     "buy and hold Sharpe": s["volatility"]["buy and hold"]["Sharpe ratio"],
     "buy and hold max drawdown": s["volatility"]["buy and hold"]["max drawdown"],
     "recent swing weekly Sharpe": s["volatility"]["recent swing strategy"]["strategy (weekly)"]["Sharpe ratio"]}
    for s in summaries
]).set_index("stock")

## 6. Accuracy against the rules

**Edge** is the model's accuracy minus the rule's accuracy for the same stock and days. The bars show the 95% range of that difference. Model and rule agree on most days, and those days add no uncertainty, so the ranges are narrow; they come from the same days as the paired test in the table below. The ranges are two-sided, while the p-values only ask whether the model is *better*, so a p-value just under 0.05 can come with a range that just touches zero.

First against "same as today", the bar from phase 4, then against Recent swing, the tougher bar:

In [ ]:
plot_edges(vol_table, "edge over same as today", "same as today",
           'Move size: model edge over "same as today", with 95% range')
plot_edges(vol_table, "edge", "recent swing", 'Move size: model edge over "recent swing", with 95% range')

## 7. Full table

- **edge / p-value vs recent swing:** the main test. The p-value is the paired test: the chance of winning this many of the days where model and rule disagree, if the model were no better.
- **edge / p-value vs same as today:** the same for the weaker rule, for context. **beats same as today (corrected)?** uses the same corrected bar.
- **Sharpe (daily / weekly):** the half-on-big strategy (full position on predicted calm days, half on predicted big days), deciding every day or once a week, after trading costs, next to buy-and-hold. The worst drop is shown for the weekly version.
- **share of random >= (daily / weekly):** from the luck checks: how often random half-position days, or random whole weeks, reached at least the same Sharpe ratio.
- **beats luck, daily / weekly:** each strategy is a separate comparison, so the corrected luck bar divides 5% by twice the number of rows.

In [ ]:
vol_table

## 8. How many wins would luck produce?

In [ ]:
luck_summary(vol_info, 'Accuracy beats "recent swing"', "Half-on-big, daily or weekly, beats luck (Sharpe ratio)")
display(Markdown(f'For context, against the weaker "same as today" rule: '
                 f'{vol_info["same as today passes (corrected)"]} of {vol_info["comparisons"]} pass after correction.'))

## 9. The LSTM against logistic regression

Only for stocks whose reports were made with `--lstm`. The same paired test as in the reports: does the LSTM, reading the raw sequence of the last 60 days, predict better than logistic regression with its five summary features, on the same days? The Holm correction counts every stock and task shown here.

In [ ]:
lstm_table, lstm_info = lstm_comparison(summaries)
if lstm_table is None:
    print("No report here was made with the LSTM. To include it: python run.py SPY NVDA AAPL GOOGL --lstm")
else:
    display(Markdown(f"{lstm_info['comparisons']} comparisons. LSTM better than logistic regression: "
                     f"{lstm_info['passes (5%)']} at 5%, {lstm_info['passes (corrected)']} after the Holm correction "
                     f"(strictest bar {lstm_info['corrected threshold']:.4f})."))
    display(lstm_table)

## 10. Conclusions

*Write this in your own words.* Some prompts:

- **Direction:** did any stock show a real edge after the correction? Were results consistent across stocks?
- **Move size:** did the models beat "same as today" on every stock after the correction? And the Recent swing rule? What does that say about where the predictability comes from?
- Did predicting move size improve the Sharpe ratio or the worst drop, after trading costs? Did deciding weekly change the answer?
- How much did buy-and-hold differ between stocks, and how much of that is hindsight from picking well-known winners?
- If the LSTM was run: did a neural network reading the raw days do any better than logistic regression?